In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split

from src.features import extract_features_batch, normalize_features, FEATURE_NAMES
from src.retrieval import DenseRetriever
from src.models import train_all_models, predict, get_feature_importance
from src.evaluate import compute_correlations

sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
import json

queries, qrels = [], {}
with open('../data/msmarco_passage/queries.dev.tsv') as f:
    for line in f:
        qid, query = line.strip().split('\t', 1)
        queries.append((qid, query))

with open('../data/msmarco_passage/qrels.dev.tsv') as f:
    for line in f:
        parts = line.strip().split()
        qid, pid, rel = parts[0], parts[2], int(parts[3])
        qrels.setdefault(qid, {})[pid] = rel

df_lookup = json.load(open('../data/msmarco_passage/df_lookup.json'))
corpus_lm = json.load(open('../data/msmarco_passage/corpus_lm.json'))

print(f'Queries: {len(queries)}')

In [ ]:
from src.evaluate import reciprocal_rank

retriever = DenseRetriever()
TOP_K = 20

all_texts, all_embs, all_q_embs, y = [], [], [], []

for qid, query in queries:
    results  = retriever.retrieve(query, top_k=TOP_K)
    rel_list = [qrels.get(qid, {}).get(r['id'], 0) for r in results]
    rr       = reciprocal_rank(rel_list, n=10)

    all_texts.append([r['text']      for r in results])
    all_embs.append( np.stack([r['embedding'] for r in results]))
    all_q_embs.append(retriever.encode_query(query))
    y.append(rr)

y = np.array(y, dtype=np.float32)
print(f'MRR@10: {y.mean():.4f}')

In [ ]:
query_strings = [q for _, q in queries]

X = extract_features_batch(
    query_strings, all_texts, all_embs,
    np.stack(all_q_embs), df_lookup, corpus_lm
)

np.save('../outputs/predictions/X_features.npy', X)
np.save('../outputs/predictions/y_mrr10.npy', y)
print(f'Features shape: {X.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_n, X_test_n, scaler = normalize_features(X_train, X_test)

models = train_all_models(X_train_n, y_train, cv=True)

In [ ]:
rows = []
for name, model in models.items():
    y_pred = predict(model, X_test_n)
    corr   = compute_correlations(y_pred, y_test)
    rows.append({
        'Model':      name,
        'Pearson r':  round(corr['pearson_r'],    4),
        'Spearman rho': round(corr['spearman_rho'], 4),
        'Kendall tau':  round(corr['kendall_tau'],  4),
        'p (Pearson)':  round(corr['pearson_p'],    4),
    })

df_corr = pd.DataFrame(rows).set_index('Model')
df_corr.to_csv('../outputs/predictions/qpp_correlations.csv')
print(df_corr)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#1E88E5', '#43A047', '#FB8C00']

for ax, (name, model), color in zip(axes, models.items(), colors):
    y_pred = predict(model, X_test_n)
    r, _   = pearsonr(y_pred, y_test)
    ax.scatter(y_pred, y_test, alpha=0.4, s=15, color=color)
    m, b = np.polyfit(y_pred, y_test, 1)
    xs = np.linspace(y_pred.min(), y_pred.max(), 100)
    ax.plot(xs, m * xs + b, 'r-', lw=2)
    ax.set_title(f'{name}  (r={r:.4f})')
    ax.set_xlabel('Predicted QPP Score')
    ax.set_ylabel('True MRR@10')

plt.tight_layout()
plt.savefig('../outputs/figures/qpp_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
imp    = get_feature_importance(models['random_forest'])
imp_df = pd.DataFrame(sorted(imp.items(), key=lambda x: x[1]),
                      columns=['Feature', 'Importance'])

def feature_color(name):
    if name in ('emb_variance', 'high_sim_count', 'sim_variance', 'rank_dropoff', 'max_sim'):
        return '#E53935'
    if name in ('term_overlap', 'query_length', 'query_idf_sum', 'query_entropy'):
        return '#78909C'
    return '#FB8C00'

plt.figure(figsize=(8, 5))
plt.barh(imp_df['Feature'], imp_df['Importance'],
         color=[feature_color(f) for f in imp_df['Feature']], edgecolor='white')
plt.xlabel('Relative Feature Importance')
plt.title('Feature Importance — Random Forest')
patches = [
    mpatches.Patch(color='#E53935', label='Semantic'),
    mpatches.Patch(color='#FB8C00', label='Traditional QPP'),
    mpatches.Patch(color='#78909C', label='Lexical'),
]
plt.legend(handles=patches)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()